# SAGE-Lite Phase 3 Ablation — Variant N (Zero Expert Branch)

**Mục tiêu:** Kiểm tra vai trò của nhánh Expert Pool trong kiến trúc SAGE.
Variant N giữ nguyên toàn bộ kiến trúc và số tham số của mô hình Canonical (10,205,755 tham số),
nhưng đặt `residual_scale = 0.0` trên tất cả `SageLayer` (đóng góp của nhánh expert = 0).

---
### Cách chạy:
- **Cách 1 (Khuyến nghị — 2 tabs song song, tiết kiệm thời gian):**
  - Tab A: Chạy **Cell 1**, **Cell 2**, **Cell 3** (Seed 42) ~ 1.75 giờ.
  - Tab B: Chạy **Cell 1**, **Cell 4**, **Cell 5** (Seed 43) ~ 1.75 giờ.
- **Cách 2 (1 tab tuần tự):** Bấm **Runtime -> Run all** (chạy lần lượt từ Cell 1 đến Cell 5) ~ 3.5 giờ.
  (Sau khi xong Cell 3, file zip Seed 42 sẽ tự động tải về trước khi chạy tiếp Seed 43).

In [ ]:
# ==============================================================================
# Cell 1: Environment Setup, Repo Clone, Dependencies & Dataset
# ==============================================================================
!nvidia-smi

!rm -rf /content/SAGE_LITE
!git clone https://github.com/bach0823/SAGE_LITE.git /content/SAGE_LITE
%cd /content/SAGE_LITE
!git log --oneline -3

!pip install -q timm einops

!python prepare_data/prepare_crack500.py --output-dir /content/dataset/Crack500

!python scripts/train_phase3_ablation.py --help | head -20

In [ ]:
# ==============================================================================
# Cell 2: Training Variant N — Seed 42 (Zero Expert Branch)
# ==============================================================================
%cd /content/SAGE_LITE

!python scripts/train_phase3_ablation.py \
    --variant N \
    --seed 42 \
    --config configs/sage_b2_crack500.yaml \
    --data-root /content/dataset/Crack500 \
    --output-dir /content/runs/phase3_N_seed42

In [ ]:
# ==============================================================================
# Cell 3: Post-Training Seed 42 — Diagnostics, Zip & Download
# ==============================================================================
%cd /content/SAGE_LITE
from google.colab import files
import shutil, os

RUN_DIR = '/content/runs/phase3_N_seed42'
CKPT    = f'{RUN_DIR}/best_model_b2_global.pth'

if os.path.exists(CKPT):
    !python tools/analyze_routing.py \
        --config configs/sage_b2_crack500.yaml \
        --checkpoint {CKPT} \
        --data-root /content/dataset/Crack500 \
        --split val \
        --output-dir {RUN_DIR}/routing_diagnostics/full_val || true

zip_path = '/content/phase3_variantN_seed42_full'
shutil.make_archive(zip_path, 'zip', '/content/runs', 'phase3_N_seed42')
print(f'Archive size: {os.path.getsize(zip_path+".zip")/1e6:.1f} MB')
files.download(zip_path + '.zip')

In [ ]:
# ==============================================================================
# Cell 4: Training Variant N — Seed 43 (Zero Expert Branch)
# ==============================================================================
%cd /content/SAGE_LITE

!python scripts/train_phase3_ablation.py \
    --variant N \
    --seed 43 \
    --config configs/sage_b2_crack500.yaml \
    --data-root /content/dataset/Crack500 \
    --output-dir /content/runs/phase3_N_seed43

In [ ]:
# ==============================================================================
# Cell 5: Post-Training Seed 43 — Diagnostics, Zip & Download
# ==============================================================================
%cd /content/SAGE_LITE
from google.colab import files
import shutil, os

RUN_DIR = '/content/runs/phase3_N_seed43'
CKPT    = f'{RUN_DIR}/best_model_b2_global.pth'

if os.path.exists(CKPT):
    !python tools/analyze_routing.py \
        --config configs/sage_b2_crack500.yaml \
        --checkpoint {CKPT} \
        --data-root /content/dataset/Crack500 \
        --split val \
        --output-dir {RUN_DIR}/routing_diagnostics/full_val || true

zip_path = '/content/phase3_variantN_seed43_full'
shutil.make_archive(zip_path, 'zip', '/content/runs', 'phase3_N_seed43')
print(f'Archive size: {os.path.getsize(zip_path+".zip")/1e6:.1f} MB')
files.download(zip_path + '.zip')